In [ ]:
!pip install -q -U langchain langchain-google-genai langchain-community faiss-cpu pypdf

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
os.environ["GOOGLE_API_KEY"]=userdata.get("GEMINI_API_KEYS")
MODEL="gemini-3.5-flash-lite"
EMB_MODEL="gemini-embedding-001"

In [ ]:
#1 Prepare Document
#1.1 Upload the document
from google.colab import files
uploaded=files.upload()
print(list(uploaded.keys()))

Saving school_faq.pdf to school_faq (1).pdf
Saving college_faq.pdf to college_faq (1).pdf
Saving placement_faq.pdf to placement_faq (1).pdf
['school_faq (1).pdf', 'college_faq (1).pdf', 'placement_faq (1).pdf']


In [ ]:
#1.2 Extract text
from langchain_community.document_loaders import PyPDFLoader
docs=[]
for file in list(uploaded.keys()):
  file_docs=PyPDFLoader(file).load()
  category = file.split("_faq")[0]
  print("CATEGORY",category)
  for d in file_docs:

    d.metadata["category"]=category
  docs.extend(file_docs)
print("NUMBER OF DOCUMENT: ",len(list(uploaded.keys())))
print("NUMEBER OF PAGES: ",len(docs))


CATEGORY school
CATEGORY college
CATEGORY placement
NUMBER OF DOCUMENT:  3
NUMEBER OF PAGES:  6


In [ ]:
#1.3Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=150)
chunks=splitter.split_documents(docs)
print("NUMBER OF CHUNKS: ",len(chunks))
text=""
for c in chunks:
  text+=c.page_content+"\n\n"
print("NUMBER OF CHARACTERS: ",len(text))
for c in chunks:
    print(c.metadata)

NUMBER OF CHUNKS:  11
NUMBER OF CHARACTERS:  6354
{'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-06T19:15:21+05:30', 'author': 'kushal yn', 'moddate': '2026-09-06T19:15:21+05:30', 'source': 'school_faq (1).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'category': 'school'}
{'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-06T19:15:21+05:30', 'author': 'kushal yn', 'moddate': '2026-09-06T19:15:21+05:30', 'source': 'school_faq (1).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'category': 'school'}
{'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-06T19:15:21+05:30', 'author': 'kushal yn', 'moddate': '2026-09-06T19:15:21+05:30', 'source': 'school_faq (1).pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'category': 'school'}
{'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-06T19:1

In [ ]:
#1.4 embedding vectorstore faiss
embeddings=GoogleGenerativeAIEmbeddings(model=EMB_MODEL)
from langchain_community.vectorstores import FAISS
vectorstore=FAISS.from_documents(chunks,embeddings)

In [ ]:
#1.5 Retreive top - k chunks
college_retriever=vectorstore.as_retriever(
    search_kwargs={'k':5,
                   'filter':{'category':'college'}})
school_retriever=vectorstore.as_retriever(
    search_kwargs={'k':5,
                   'filter':{'category':'school'}}
)
placement_retriever=vectorstore.as_retriever(
    search_kwargs={'k':5,
                   'filter':{'category':'placement'}}
)
print("RETRIEVERS READY")


RETRIEVERS READY


In [ ]:
#LLM RAG_CHAIN CREATION
#2.Answer and formatting answer
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
llm=ChatGoogleGenerativeAI(model=MODEL,
                            max_output_tokens=1500)
prompt=ChatPromptTemplate.from_messages([
    ('system',"""You are a PDF assistant.

The user has selected a category before asking the question.

Answer ONLY using the information present in the provided document context.

Important:
- Match the user's question by MEANING, not only by exact words.
- Treat synonyms and different ways of asking the same thing as equivalent.
  Example:
  "Where do I collect my ID card?"
  and
  "Where are identity cards issued?"
  can refer to the same information.
- Use the context to determine the answer even when the words in the question
  are different from the words in the document.
- Do not use outside knowledge or invent information.

Rules:
1. If the question is relevant to the selected category and the context contains
   relevant information, answer using that information.

2. If the question is clearly about another category, say:
   "This question is outside the selected category. Please choose the appropriate category."

3. If the question belongs to the selected category but the provided context
   does not contain enough information, say:
   "I could not find enough information in the selected document."

4. If the user asks for a summary, summarize only the information available
   in the provided context.
"""),
    ('human',"context:{context}\n\n{Question}")

])
def format_docs(docs):
  return ("\n\n---------------\n\n".join(d.page_content for d in docs))
user_category=int(input("1 for School\n2 for College\n3 for Placement\nChoose the category where you have query by corresponding number"))

valid=1
if user_category==1:
  retriever=school_retriever
elif user_category==2:
  retriever=college_retriever
elif user_category==3:
  retriever=placement_retriever
else:
  print("INVALID CATEGORY")
  valid=0
if valid:
  rag_chain=(
    {'context':retriever|format_docs,'Question':RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
  )
  print("READY!!!!")

1 for School
2 for College
3 for Placement
Choose the category where you have query by corresponding number1
READY!!!!


In [48]:

print("NOTE ENTER bye,exit,quit to terminate session")
while True:
  user_input=input("ENTER YOU Query: ")
  if user_input.lower() in ['exit','bye','quit']:
    print("EXITED")
    break
  response=rag_chain.invoke(user_input)
  print(f"AI RESPONSE:{response}")

NOTE ENTER bye,exit,quit to terminate session
ENTER YOU Query: i lost my id card where to collect
AI RESPONSE:Students can obtain their school ID card through the school administration.
ENTER YOU Query: tell about leave policy and where can parents apply leave on behalf of students
AI RESPONSE:Parents can submit a leave request through the school's prescribed leave application process.
ENTER YOU Query: leave policy???
AI RESPONSE:I could not find enough information in the selected document.
ENTER YOU Query: summarize
AI RESPONSE:Based on the provided document, here is a summary of the information:

* **School Hours:** The school operates from 8:00 AM to 4:00 PM.
* **Student Attendance & Leave:** Parents can check their child's attendance via the school's attendance system and apply for leave using the prescribed leave application process.
* **Examination Results:** Students can check their results through the school's result system or by contacting the school administration.
* **School